# 스트레스 점수 예측 V10

**V9 Pooled Quantile Forest + Age/Health Interaction Ensemble**

- V9: 3개 시드, 총 3,600개 ExtraTrees 예측의 50.5% 분위수
- 보조 모델: 나이·건강지표 상호작용 피처를 사용한 ExtraTrees
- 최종 예측: V9 70% + 보조 모델 30%
- MAE에 맞춰 최종 예측을 소수점 둘째 자리로 반올림
- 모든 전처리 기준은 Train 데이터에서만 학습

In [1]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [2]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
SEEDS = (42, 77, 2026)
N_ESTIMATORS = 1200
V9_QUANTILE = 0.505
AGE_QUANTILE = 0.50
V9_WEIGHT = 0.70
AGE_WEIGHT = 0.30

V9_WEIGHTED_FEATURES = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

AGE_WEIGHTED_FEATURES = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
    "mean_arterial_pressure",
    "diastolic_blood_pressure",
]

## 2. 데이터 불러오기 및 검증

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
if train[TARGET].isna().any():
    raise ValueError("Train 타깃에 결측값이 있습니다.")

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

Train: (3000, 18)
Test: (3000, 17)


,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


## 3. 두 모델의 행 단위 파생변수

In [4]:
def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_v9_features(frame):
    result = add_base_features(frame)
    for feature in V9_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


def add_age_features(frame):
    result = add_base_features(frame)
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_glucose"] = result["age"] * result["glucose"]
    result["age_cholesterol"] = result["age"] * result["cholesterol"]
    result["age_bone_density"] = result["age"] * result["bone_density"]
    result["age_mean_arterial_pressure"] = (
        result["age"] * result["mean_arterial_pressure"]
    )
    for feature in AGE_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
    return result


raw_train = train.drop(columns=[TARGET, ID_COLUMN])
raw_test = test.drop(columns=[ID_COLUMN])
y_train = train[TARGET]

X_v9 = add_v9_features(raw_train)
X_test_v9 = add_v9_features(raw_test)
X_age = add_age_features(raw_train)
X_test_age = add_age_features(raw_test)

print("V9 features:", X_v9.shape)
print("Age features:", X_age.shape)

V9 features: (3000, 38)
Age features: (3000, 37)


## 4. 누수 없는 전처리 함수

In [5]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
                numerical,
            ),
        ]
    )


def tree_prediction_matrix(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )

## 5. 5-Fold 교차검증

In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mae = []
oof_v9 = np.zeros(len(X_v9))
oof_age = np.zeros(len(X_v9))
oof_v10 = np.zeros(len(X_v9))

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_v9),
    start=1,
):
    # V9: Fold Train에서만 전처리 학습
    v9_preprocessor = make_preprocessor(X_v9.iloc[train_idx])
    v9_train_matrix = v9_preprocessor.fit_transform(
        X_v9.iloc[train_idx]
    )
    v9_valid_matrix = v9_preprocessor.transform(
        X_v9.iloc[valid_idx]
    )

    v9_tree_predictions = []
    for seed in SEEDS:
        model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(v9_train_matrix, y_train.iloc[train_idx])
        v9_tree_predictions.append(
            tree_prediction_matrix(model, v9_valid_matrix)
        )

    v9_prediction = np.quantile(
        np.concatenate(v9_tree_predictions, axis=1),
        V9_QUANTILE,
        axis=1,
    )
    oof_v9[valid_idx] = np.clip(v9_prediction, 0.0, 1.0)

    # 나이·건강 상호작용 보조 모델
    age_preprocessor = make_preprocessor(X_age.iloc[train_idx])
    age_train_matrix = age_preprocessor.fit_transform(
        X_age.iloc[train_idx]
    )
    age_valid_matrix = age_preprocessor.transform(
        X_age.iloc[valid_idx]
    )
    age_model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=42,
        n_jobs=-1,
    )
    age_model.fit(age_train_matrix, y_train.iloc[train_idx])
    age_prediction = np.quantile(
        tree_prediction_matrix(age_model, age_valid_matrix),
        AGE_QUANTILE,
        axis=1,
    )
    oof_age[valid_idx] = np.clip(age_prediction, 0.0, 1.0)

    # MAE에 맞춘 가중 앙상블과 소수점 둘째 자리 반올림
    valid_prediction = np.round(
        V9_WEIGHT * oof_v9[valid_idx]
        + AGE_WEIGHT * oof_age[valid_idx],
        2,
    )
    valid_prediction = np.clip(valid_prediction, 0.0, 1.0)
    oof_v10[valid_idx] = valid_prediction

    score = mean_absolute_error(
        y_train.iloc[valid_idx],
        valid_prediction,
    )
    fold_mae.append(score)
    print(f"Fold {fold} MAE: {score:.6f}")

fold_mae = np.array(fold_mae)
v10_cv_mae = mean_absolute_error(y_train, oof_v10)

print("V10 Fold MAE:", np.round(fold_mae, 6))
print("V9 Mean MAE:", "0.148685")
print("V10 Mean MAE:", f"{v10_cv_mae:.6f}")
print("V10 MAE Std:", f"{fold_mae.std():.6f}")
print("V10 vs V9:", f"{v10_cv_mae - 0.148685:+.6f}")

Fold 1 MAE: 0.135533
Fold 2 MAE: 0.151083
Fold 3 MAE: 0.148033
Fold 4 MAE: 0.158400
Fold 5 MAE: 0.149683
V10 Fold MAE: [0.135533 0.151083 0.148033 0.1584   0.149683]
V9 Mean MAE: 0.148685
V10 Mean MAE: 0.148547
V10 MAE Std: 0.007408
V10 vs V9: -0.000138


## 6. 전체 Train 학습 및 Test 예측

In [7]:
# V9 전체 학습
v9_preprocessor = make_preprocessor(X_v9)
v9_train_matrix = v9_preprocessor.fit_transform(X_v9)
v9_test_matrix = v9_preprocessor.transform(X_test_v9)

v9_test_tree_predictions = []
for seed in SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(v9_train_matrix, y_train)
    v9_test_tree_predictions.append(
        tree_prediction_matrix(model, v9_test_matrix)
    )
    print(f"V9 seed {seed}: complete")

pred_v9 = np.quantile(
    np.concatenate(v9_test_tree_predictions, axis=1),
    V9_QUANTILE,
    axis=1,
)

# 나이·건강 상호작용 모델 전체 학습
age_preprocessor = make_preprocessor(X_age)
age_train_matrix = age_preprocessor.fit_transform(X_age)
age_test_matrix = age_preprocessor.transform(X_test_age)

age_model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=1,
    max_features=1,
    random_state=42,
    n_jobs=-1,
)
age_model.fit(age_train_matrix, y_train)
pred_age = np.quantile(
    tree_prediction_matrix(age_model, age_test_matrix),
    AGE_QUANTILE,
    axis=1,
)
print("Age interaction model: complete")

V9 seed 42: complete
V9 seed 77: complete
V9 seed 2026: complete
Age interaction model: complete


## 7. 최종 예측값 확인

In [8]:
pred_v10 = np.round(
    V9_WEIGHT * pred_v9 + AGE_WEIGHT * pred_age,
    2,
)
pred_v10 = np.clip(pred_v10, 0.0, 1.0)

print("Prediction shape:", pred_v10.shape)
print("Prediction range:", pred_v10.min(), pred_v10.max())
print("Prediction mean:", pred_v10.mean())
print("첫 10개 예측값:")
print(pred_v10[:10])

Prediction shape: (3000,)
Prediction range: 0.0 1.0
Prediction mean: 0.49868
첫 10개 예측값:
[0.57 0.97 0.19 0.44 0.53 0.46 0.78 0.46 0.56 0.88]


## 8. 제출 파일 생성 및 확인

In [9]:
submission_v10 = submission.copy()
submission_v10[ID_COLUMN] = test[ID_COLUMN]
submission_v10[TARGET] = pred_v10

output_path = OUTPUT_DIR / "submit_v10_v9_age_ensemble.csv"
submission_v10.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path)
print("Rows:", len(submission_v10))
submission_v10.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v10_v9_age_ensemble.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.57
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.44
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.56
9,TEST_0009,0.88


### 예측값 요약 통계

In [10]:
submission_v10[TARGET].describe().to_frame(name="stress_score")

,stress_score
count,3000.00000
mean,0.49868
std,0.20295
min,0.00000
25%,0.41000
50%,0.50000
75%,0.57000
max,1.00000
